
[Variant Blog Post](https://www.databricks.com/blog/introducing-variant-new-open-standard-semi-structured-data-apache-parquettm-delta-lake)

In [0]:
%pip install dbldatagen

In [0]:
from pyspark.sql.types import LongType, FloatType, IntegerType, StringType, \
                              DoubleType, BooleanType, ShortType, \
                              TimestampType, DateType, DecimalType, \
                              ByteType, BinaryType, ArrayType, MapType, \
                              StructType, StructField

import dbldatagen as dg

device_population = 10000000

country_codes = ['CN', 'US', 'FR', 'CA', 'IN', 'JM', 'IE', 'PK', 'GB', 'IL', 'AU', 'SG',
                 'ES', 'GE', 'MX', 'ET', 'SA', 'LB', 'NL']
country_weights = [1300, 365, 67, 38, 1300, 3, 7, 212, 67, 9, 25, 6, 47, 83, 126, 109, 58, 8,
                   17]

manufacturers = ['Delta corp', 'Xyzzy Inc.', 'Lakehouse Ltd', 'Acme Corp', 'Embanks Devices']

lines = ['delta', 'xyzzy', 'lakehouse', 'gadget', 'droid']

testDataSpec = (
    dg.DataGenerator(spark, name="device_data_set", rows=100000000,
                     partitions=8, randomSeedMethod='hash_fieldname')
    .withIdOutput()
    # we'll use hash of the base field to generate the ids to
    # avoid a simple incrementing sequence
    .withColumn("internal_device_id", LongType(), minValue=0x1000000000000,
                uniqueValues=device_population, omit=True, baseColumnType="hash")

    # note for format strings, we must use "%lx" not "%x" as the
    # underlying value is a long
    .withColumn("device_id", StringType(), format="0x%013x",
                baseColumn="internal_device_id")

    # the device / user attributes will be the same for the same device id
    # so lets use the internal device id as the base column for these attribute
    .withColumn("country", StringType(), values=country_codes,
                weights=country_weights,
                baseColumn="internal_device_id")

    .withColumn("manufacturer", StringType(), values=manufacturers,
                baseColumn="internal_device_id", omit=True)
    .withColumn("line", StringType(), values=lines, baseColumn="manufacturer",
                baseColumnType="hash", omit=True)
    .withColumn("manufacturer_info", StructType([StructField('line',StringType()),
                                                StructField('manufacturer', StringType())]),
                expr="named_struct('line', line, 'manufacturer', manufacturer)",
                baseColumn=['manufacturer', 'line'])


    .withColumn("model_ser", IntegerType(), minValue=1, maxValue=11,
                baseColumn="device_id",
                baseColumnType="hash", omit=True)

    .withColumn("event_type", StringType(),
                values=["activation", "deactivation", "plan change",
                        "telecoms activity", "internet activity", "device error"],
                random=True, omit=True)
    .withColumn("event_ts", "timestamp", begin="2020-01-01 01:00:00",
                end="2020-12-31 23:59:00",
                interval="1 minute", random=True, omit=True)

    .withColumn("event_info",
                 StructType([StructField('event_type',StringType()),
                             StructField('event_ts', TimestampType())]),
                expr="named_struct('event_type', event_type, 'event_ts', event_ts)",
                baseColumn=['event_type', 'event_ts'])
    )

dfTestData = testDataSpec.build()
dfTestData.write.format("json").mode("overwrite").save("/Volumes/usa/devices/data")



In [0]:
df_string = spark.read.format("text").load("/Volumes/usa/devices/data")
display(df_string)

In [0]:
df_string.createOrReplaceTempView("json_string")

## Create a Variant column

```
CREATE TABLE T (variant_col Variant)
```

## Use PARSE_JSON() to parse JSON string to Variant

```
INSERT INTO T (variant_col)
SELECT PARSE_JSON(json_str_col) FROM other_table
```


In [0]:
%sql

CREATE OR REPLACE TABLE usa.devices.devices AS
  SELECT parse_json(value) as devices
  FROM json_string

In [0]:
%sql
SELECT COUNT(*)
FROM usa.devices.devices

In [0]:
%sql
SELECT *
FROM usa.devices.devices
LIMIT 1

## Query fields in a variant column

The syntax for querying JSON strings and other complex data types on Databricks applies to VARIANT data, including the following:

- Use : to select top level fields.
- Use . or [<key>] to select nested fields with named keys.
- Use [<index>] to select values from arrays.

## Path navigation

```sql
SELECT 
  variant_col:a.b.c::int, 
  variant_col:arr[1].field::double 
FROM T
```

## Flatten objects
```
SELECT key, value 
FROM T, LATERAL explode(T.variant_col:obj)
```

In [0]:
%sql
SELECT devices:country, devices
FROM usa.devices.devices


In [0]:
%sql
SELECT devices:event_info.event_type, devices
FROM usa.devices.devices

In [0]:
%sql
SELECT country, COUNT(country)
FROM (
    SELECT string(devices:country) as country
    FROM usa.devices.devices
    )
GROUP BY ALL


In [0]:
%sql
CREATE OR REPLACE TABLE usa.devices.device_events 
CLUSTER BY AUTO AS
  SELECT string(devices:country) as country, string(devices:device_id) as device_id, devices:event_info as event_info, devices
  FROM usa.devices.devices


In [0]:
%sql
SELECT COUNT(*)
FROM usa.devices.device_events

In [0]:
%sql
SELECT country, COUNT(country)
FROM usa.devices.device_events
GROUP BY ALL

In [0]:
%sql
SELECT *
FROM usa.devices.device_events
WHERE device_id = "0x10000007c3888"